In [ ]:
#| hide
#| eval: false
! [ -e /content ] && pip install -Uqq fastai  # upgrade fastai on colab

In [ ]:
from fastai.text.all import *
from fastai.text.tokenizer import *

# Text transfer learning

> How to fine-tune a language model and train a classifier

In this tutorial, we will see how we can train a model to classify text (here based on their sentiment). First we will see how to do this quickly in a few lines of code, then how to get state-of-the art results using the approach of the [ULMFit paper](https://arxiv.org/abs/1801.06146).

We will use the IMDb dataset from the paper [Learning Word Vectors for Sentiment Analysis](https://ai.stanford.edu/~amaas/data/sentiment/), containing a few thousand movie reviews.

## Train a text classifier from a pretrained model

We will try to train a classifier using a pretrained model, a bit like we do in the [vision tutorial](23_tutorial.vision.ipynb). To get our data ready, we will first use the high-level API:

## Using the high-level API

We can download the data and decompress it with the following command:

In [ ]:
path = untar_data(URLs.IMDB)
path.ls()

[Path('/Users/pengren/.fastai/data/imdb/test'), Path('/Users/pengren/.fastai/data/imdb/tmp_clas'), Path('/Users/pengren/.fastai/data/imdb/imdb.vocab'), Path('/Users/pengren/.fastai/data/imdb/unsup'), Path('/Users/pengren/.fastai/data/imdb/models'), Path('/Users/pengren/.fastai/data/imdb/README'), Path('/Users/pengren/.fastai/data/imdb/tmp_lm'), Path('/Users/pengren/.fastai/data/imdb/train')]

In [ ]:
(path/'train').ls()

[Path('/Users/pengren/.fastai/data/imdb/train/neg'), Path('/Users/pengren/.fastai/data/imdb/train/pos'), Path('/Users/pengren/.fastai/data/imdb/train/unsupBow.feat'), Path('/Users/pengren/.fastai/data/imdb/train/labeledBow.feat')]

The data follows an ImageNet-style organization, in the train folder, we have two subfolders, `pos` and `neg` (for positive reviews and negative reviews). We can gather it by using the `TextDataLoaders.from_folder` method. The only thing we need to specify is the name of the validation folder, which is "test" (and not the default "valid").

In [ ]:
# sample_path = untar_data(URLs.IMDB_SAMPLE)
# dls = TextDataLoaders.from_csv(sample_path, 'texts.csv', text_col='text', label_col='label', valid_col='is_valid', bs=8)

In [ ]:
dls = TextDataLoadersNew.from_dataset('stanfordnlp/imdb', 'text', get_y='label', valid_pct=0.2, seed=42, bs=8, split='train[:500]')

We can then have a look at the data with the `show_batch` method:

In [ ]:
dls.show_batch()

,text,category
0,"<bos>If you've ever been harassed on the Underground by a Christian who says, ""Jesus is the answer. What's the question?"", then perhaps you should thank God if you've never met a Lacanian. Slavoj Zizek, the most evangelical of Lacanians, would surely exchange the word ""Jesus"" in that statement for ""Lacan/Hegel"".<br /><br />Zizek's star burns brightly at the moment, no doubt because we generally view films and pop culture purely as entertainment for our consumption. So it seems impressive when someone - anyone - comes along and says, ""Hang on, films may say something about ourselves.""<br /><br />The ideas Zizek expounds in this film are ""true"" purely because he says so. For example, Zizek explains that three Marx Bros are the ego, superego and id (God knows what happened to Zeppo, or Gummo perhaps they're the sinthome...or is that movies themselves?). This is simply what they are. In Zizek's output,",0
1,"<bos>Within the realm of Science Fiction, two particular themes consistently elicit interest, were initially explored in the literature of a pre-cinematic era, and have since been periodically revisited by filmmakers and writers alike, with varying degrees of success. The first theme, that of time travel, has held an unwavering fascination for fans of film, as well as the written word, most recently on the screen with yet another version of the H.G. Wells classic, `The Time Machine.' The second theme, which also manages to hold audiences in thrall, is that of invisibility, which sparks the imagination with it's seemingly endless and myriad possibilities. And this theme, too, has again become the basis for a film adapted from another H.G. Wells classic, `The Invisible Man,' the realization of which, here, is `Hollow Man,' directed by Paul Verhoeven, and starring Kevin Bacon and Elisabeth Shue.<br /><br />Sebastian Caine (Bacon) and his colleagues",0
2,"<bos>""Spielberg loves the smell of sentiment in the morning. But sentiment at the expense of narrative honesty? Nobody should love that."" - Lucius Shepard<br /><br />""The Color Purple"" takes place in the Deep South during the early 1900s, and tells the story of Celie and Nettie, two African American sisters. The film opens with the girls playing in a field of purple flowers, an idyllic haven which is promptly shattered by the appearance of their stepfather. This motif  innocence interrupted by men  permeates the entire film.<br /><br />The film then launches into a series of short sequences. Celie is revealed to have been twice impregnated by her stepfather, gives birth in a dirty barn, has her newborn child taken away and is forced to marry a local widow named Albert Johnson, a violent oaf who rapes her repeatedly, forcing her to cook, clean and look after his children.<br",0
3,"<bos>I have this film out of the library right now and I haven't finished watching it. It is so bad I am in disbelief. Audrey Hepburn had totally lost her talent by then, although she'd pretty much finished with it in 'Robin and Marian.' This is the worst thing about this appallingly stupid film. It's really only of interest because it was her last feature film and because of the Dorothy Stratten appearance just prior to her homicide.<br /><br />There is nothing but idiocy between Gazzara and his cronies. Little signals and little bows and nods to real screwball comedy of which this is the faintest, palest shadow.<br /><br />Who could believe that there are even some of the same Manhattan environs that Hepburn inhabited so magically and even mythically in 'Breakfast at Tiffany's' twenty years earlier? The soundtrack of old Sinatra songs and the Gershwin song from which the",0
4,"<bos>Well, as Goethe once said, there really isn't any point in trying to pass a negative judgement that aspires to be objective on ""something that has had a great effect"". ""La Maman et La Putain"" has surely passed into history as an influence on much of what's been done in France and elsewhe

We can see that the library automatically processed all the texts to split then in *tokens*, adding some special tokens like:

- `<bos>` to indicate the beginning of a text

Then, we can define a `Learner` suitable for text classification in one line:

In [ ]:
learn = text_classifier_learner(dls, AWD_LSTM, drop_mult=0.5, metrics=accuracy)

We use the [AWD LSTM](https://arxiv.org/abs/1708.02182) architecture, `drop_mult` is a parameter that controls the magnitude of all dropouts in that model, and we use `accuracy` to track down how well we are doing. We can then fine-tune our pretrained model:

In [ ]:
learn.fine_tune(1, 1e-2)

epoch     train_loss  valid_loss  accuracy  time    
0         0.000000    0.000000    1.000000  00:06     
epoch     train_loss  valid_loss  accuracy  time    
0         0.000000    0.000000    1.000000  00:11     


In [ ]:
# learn.fine_tune(4, 1e-2)

Not too bad! To see how well our model is doing, we can use the `show_results` method:

In [ ]:
learn.show_results()

,text,category,category_
0,"<bos>Already his first claim, that desires are always artificial, is totally fallacious.<br /><br />When a Jehovah Witness reject gets his own documentary on movies  or anything for that matter - it's time for anyone to get their own. Although far, far more intelligent than, say, Paris Hilton (I know, not too difficult) Zizek's mouth spews just as much baloney as hers, just a different kind. He combines the worst from both his professional worlds: psychoanalysis and philosophy. Both fields are notorious for conveniently offering the expert b*lls***osopher plenty of leeway to create unprovable theories, to rant without a beginning or end, and to connect concepts almost randomly, in the process misusing the English language by creating a semantic jumble only a mother can love. Example: there are three main Marx brothers hence what a ""great"" idea to connect them with three levels of human consciousness, the id, the ego",0,0
1,"<bos>I didn't know whether to laugh or cry at this misrepresentation of Canadian history, particularly the disservice done to the history of the Mounted Police in the Yukon.<br /><br />I'll leave it to Pierre Berton, noted historian, born and raised in Dawson City Yukon, and author of the definitive history of the Klondike gold rush, Klondike: The Last Great Gold Rush, 1896-1899 to express my exasperation with this silly movie: <br /><br />The American idea of an untamed frontier, subdued by individual heroes armed with six-guns, was continued in The Far Country, another story about a cowboy from the American west - Wyoming this time - driving his herd of beef cattle into gold country. The picture is a nightmare of geographical impossibilities, but the real incongruity is the major assumption on which the plot turns  that there was only one mounted policeman in all of the Canadian Yukon",0,0
2,"<bos>WARNING: This review contains SPOILERS. Do not read if you don't want some points revealed to you before you watch the film.<br /><br />With a cast like this, you wonder whether or not the actors and actresses knew exactly what they were getting into. Did they see the script and say, `Hey, Close Encounters of the Third Kind was such a hit that this one can't fail.' Unfortunately, it does. Did they even think to check on the director's credentials? I mean, would YOU do a movie with the director of a movie called `Satan's Cheerleaders?' Greydon Clark, who would later go on to direct the infamous `Final Justice,' made this. It makes you wonder how the people of Mystery Science Theater 3000 could hammer `Final Justice' and completely miss out on `The Return.'<br /><br />The film is set in a small town in New Mexico. A little boy and",0,0
3,"<bos>When an attempt is made to assassinate the Emir of Ohtar, an Arab potentate visiting Washington, D.C., his life is saved by a cocktail waitress named Sunny Davis. Sunny becomes a national heroine and media celebrity and as a reward is offered a job working for the Protocol Section of the United States Department of State. Unknown to her however, the State Department officials who offer her the job have a hidden agenda.<br /><br />A map we see shows Ohtar lying on the borders of Saudi Arabia and South Yemen, in an area of barren desert known as the Rub al-Khali, or Empty Quarter. In real life a state in this location would have a population of virtually zero, and virtually zero strategic value, but for the purposes of the film we have to accept that Ohtar is of immense strategic importance in the Cold War and that the American",0,0
4,"<bos>I get to the cinema every week or so, and regularly check out this site, but never before have I felt compelled to comment on a film.<br /><br />To my all time list of shockingly bad films - Last Man Standing, Spawn, The Bone Collector - I can now add the drivel that was 'Hollow Man'.<br /><br />From the awful opening titles - a ridiculously over-long run through of cast and crew put together with alphabetti 

And we can predict on new texts quite easily:

In [ ]:
learn.predict("I really liked that movie!")

('0', tensor(0), tensor([1.]))

Here we can see the model has considered the review to be positive. The second part of the result is the index of "pos" in our data vocabulary and the last part is the probabilities attributed to each class (99.1% for "pos" and 0.9% for "neg"). 

Now it's your turn! Write your own mini movie review, or copy one from the Internet, and we can see what this model thinks about it. 

### Using the data block API

We can also use the data block API to get our data in a `DataLoaders`. This is a bit more advanced, so feel free to skip this part if you are not comfortable with learning new APIs just yet.

A datablock is built by giving the fastai library a bunch of information:

- the types used, through an argument called `blocks`: here we have texts and categories, so we pass `TextBlock` and `CategoryBlock`. To inform the library our texts are files in a folder, we use the `from_folder` class method.
- how to get the raw items, here our function `get_text_files`.
- how to label those items, here with the parent folder.
- how to split those items, here with the grandparent folder.

In [ ]:
# imdb = DataBlock(blocks=(HfTextBlock.from_folder(path), CategoryBlock),
#                  get_items=get_text_files,
#                  get_y=parent_label,
#                  splitter=GrandparentSplitter(valid_name='test'))
path = untar_data(URLs.IMDB)
files, docs, tok_imdb = tokenize_folder(path)
files[0]

Path('/Users/pengren/.fastai/data/imdb/test/neg/0_2.txt')

In [ ]:
path

Path('/Users/pengren/.fastai/data/imdb')

In [ ]:
!ls /Users/pengren/.fastai/data/imdb/train

labeledBow.feat neg             pos             unsupBow.feat


In [ ]:
dls = DataBlock(blocks=(HfTextBlock(HfTokenizer(hf_tokenizer()), seq_len=3), CategoryBlock),
                # get_y=parent_label,
                splitter=RandomSplitter(0.5, seed=42))

In [ ]:
                
                .dataloaders(files, bs=1)
x, y = dls.one_batch()
x.shape, len(dls.vocab), dls.pad_id

In [ ]:
%debug

> /Users/pengren/ws/fastai/fastai/data/block.py(147)datasets()
    145     ) -> Datasets:
    146         self.source = source                     ; pv(f"Collecting items from {source}", verbose)
--> 147         items = (self.get_items or noop)(source) ; pv(f"Found {len(items)} items", verbose)
    148         splits = (self.splitter or RandomSplitter())(items)
    149         pv(f"{len(splits)} datasets of sizes {','.join([str(len(s)) for s in splits])}", verbose)

<function docs>


In [ ]:
%%prompt
how to use a subset of the data in the above example too?

I’ll check the DataBlock API for selecting a subset before tokenization, rather than processing the whole folder.The docs separate folder and DataFrame tokenization. I’ll check the DataFrame example so we can reuse your downloaded sample CSV.Reuse your existing **`sample_path`**, switching from folder-based text to the sample CSV. Fastai documents this DataBlock approach directly. ([docs.fast.ai](https://docs.fast.ai/text.data.html))

```python
df = pd.read_csv(sample_path/'texts.csv')

imdb = DataBlock(
    blocks=(TextBlock.from_df('text'), CategoryBlock),
    get_x=ColReader('text'),
    get_y=ColReader('label'),
    splitter=ColSplitter('is_valid'))

dls = imdb.dataloaders(df, bs=8)
```

`ColSplitter('is_valid')` uses the CSV’s existing train/validation assignments instead of folder names. ([docs.fast.ai](https://docs.fast.ai/data.transforms.html))

The important difference: your traceback shows that **`TextBlock.from_folder(path)` starts tokenizing during construction**, before `get_items` runs. Restricting `get_items` alone therefore wouldn’t avoid that initial work. Here, tokenization operates on the sample DataFrame instead. ([docs.fast.ai](https://docs.fast.ai/text.core.html))

```json {.tool}
{
  "id": "srv_1387c7cf2a31a074b94ce917246514f0",
  "name": "web_search",
  "args": {
    "type": "open_page",
    "url": "https://docs.fast.ai/text.data.html"
  },
  "result": "Text data – fastai: https://docs.fast.ai/text.data.html\nData transformations – fastai: https://docs.fast.ai/data.transforms.html\nText core – fastai: https://docs.fast.ai/text.core.html",
  "server": true,
  "md": true
}
```

```json {.tool}
{
  "id": "srv_38c247df5ea104b50e53c83b685137d1",
  "name": "web_search",
  "args": {
    "type": "open_page",
    "url": "https://docs.fast.ai/text.data.html"
  },
  "result": "Text data – fastai: https://docs.fast.ai/text.data.html\nData transformations – fastai: https://docs.fast.ai/data.transforms.html\nText core – fastai: https://docs.fast.ai/text.core.html",
  "server": true,
  "md": true
}
```

```json {.usage}
{"model": "openai/gpt-6-astra", "prompt_tokens": 67112, "completion_tokens": 1394, "total_tokens": 68506, "reasoning_tokens": 1099, "web_search_requests": 2, "cost": 1.1412300000000002}
```


In [ ]:
df = pd.read_csv(sample_path/'texts.csv')

imdb = DataBlock(
    blocks=(TextBlock.from_df('text'), CategoryBlock),
    get_x=ColReader('text'),
    get_y=ColReader('label'),
    splitter=ColSplitter('is_valid'))

This only gives a blueprint on how to assemble the data. To actually create it, we need to use the `dataloaders` method:

In [ ]:
dls = imdb.dataloaders(df, bs=8)

In [ ]:
# dls = imdb.dataloaders(path)

## The ULMFiT approach

The pretrained model we used in the previous section is called a language model. It was pretrained on Wikipedia on the task of guessing the next word, after reading all the words before. We got great results by directly fine-tuning this language model to a movie review classifier, but with one extra step, we can do even better: the Wikipedia English is slightly different from the IMDb English. So instead of jumping directly to the classifier, we could fine-tune our pretrained language model to the IMDb corpus and *then* use that as the base for our classifier.

One reason, of course, is that it is helpful to understand the foundations of the models that you are using. But there is another very practical reason, which is that you get even better results if you fine tune the (sequence-based) language model prior to fine tuning the classification model. For instance, in the IMDb sentiment analysis task, the dataset includes 50,000 additional movie reviews that do not have any positive or negative labels attached in the unsup folder. We can use all of these reviews to fine tune the pretrained language model — this will result in a language model that is particularly good at predicting the next word of a movie review. In contrast, the pretrained model was trained only on Wikipedia articles.

The whole process is summarized by this picture:

![ULMFit process](images/ulmfit.png)

### Fine-tuning a language model on IMDb

We can get our texts in a `DataLoaders` suitable for language modeling very easily:

In [ ]:
sample_path = untar_data(URLs.IMDB_SAMPLE)
dls_lm = TextDataLoaders.from_csv(sample_path, 'texts.csv',is_lm=True, text_col='text', label_col='label', valid_col='is_valid', bs=8)

# dls_lm = TextDataLoaders.from_folder(sample_path, is_lm=True, valid_pct=0.1)

We need to pass something for `valid_pct` otherwise this method will try to split the data by using the grandparent folder names. By passing `valid_pct=0.1`, we tell it to get a random 10% of those reviews for the validation set.

We can have a look at our data using `show_batch`. Here the task is to guess the next word, so we can see the targets have all shifted one word to the right.

In [ ]:
dls_lm.show_batch(max_n=5)

xxbos xxmaj rock n ' roll is a messy business and dig ! xxunk this masterfully . a project of serious ambition , and perhaps xxunk , the filmmaker is able to xxunk together seven xxunk years of following around two xxunk rock groups . xxmaj with that said , the xxunk of quality material xxunk the film 's ability to xxunk the audience . xxmaj if you 've ever been interested
other characters were well cast from xxmaj jeff xxmaj bridges to xxmaj danny xxmaj xxunk and xxmaj gillian xxmaj anderson . xxmaj surprisingly , xxmaj xxunk xxmaj xxunk in my opinion xxunk well in this movie as the love angle to xxmaj pegg 's character however , the xxunk that i saw in xxmaj interview with the xxmaj vampire is still lost . xxmaj she needs to find it , fast or
xxmaj his character , xxmaj xxunk , must have become king of the xxmaj danes only minutes before the film opened and had n't had a chance to get the crown xxunk to fit him yet . 

 - xxmaj to xxunk the actors ' return to their day jobs waiting t

Then we have a convenience method to directly grab a `Learner` from it, using the `AWD_LSTM` architecture like before. We use accuracy and perplexity as metrics (the later is the exponential of the loss) and we set a default weight decay of 0.1. `to_fp16` puts the `Learner` in mixed precision, which is going to help speed up training on GPUs that have Tensor Cores.

In [ ]:
learn = language_model_learner(dls_lm, AWD_LSTM, metrics=[accuracy, Perplexity()], path=path, wd=0.1).to_fp16()

By default, a pretrained `Learner` is in a frozen state, meaning that only the head of the model will train while the body stays frozen. We show you what is behind the fine_tune method here and use a fit_one_cycle method to fit the model:

In [ ]:
learn.fit_one_cycle(1, 1e-2)

/Users/pengren/ws/fastai/fastai/callback/fp16.py:48: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  self.autocast,self.learn.scaler,self.scales = autocast('cuda', dtype=dtype),GradScaler('cuda', **self.kwargs),L()
/Users/pengren/ws/fastai/fastai/callback/fp16.py:48: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  self.autocast,self.learn.scaler,self.scales = autocast('cuda', dtype=dtype),GradScaler('cuda', **self.kwargs),L()


epoch     train_loss  valid_loss  accuracy  perplexity  time    
0         4.348186    4.062644    0.273992  58.127827   00:23     


This model takes a while to train, so it's a good opportunity to talk about saving intermediary results. 

You can easily save the state of your model like so:

In [ ]:
learn.save('1epoch')

Path('/Users/pengren/.fastai/data/imdb/models/1epoch.pth')

It will create a file in `learn.path/models/` named "1epoch.pth". If you want to load your model on another machine after creating your `Learner` the same way, or resume training later, you can load the content of this file with:

In [ ]:
learn = learn.load('1epoch')

We can them fine-tune the model after unfreezing:

In [ ]:
learn.unfreeze()
learn.fit_one_cycle(10, 1e-3)

epoch,train_loss,valid_loss,accuracy,perplexity,time
0,3.893486,3.772820,0.317104,43.502548,12:37
1,3.820479,3.717197,0.323790,41.148880,12:30
2,3.735622,3.659760,0.330321,38.851997,12:09
3,3.677086,3.624794,0.333960,37.516987,12:12
4,3.636646,3.601300,0.337017,36.645859,12:05
5,3.553636,3.584241,0.339355,36.026001,12:04
6,3.507634,3.571892,0.341353,35.583862,12:08
7,3.444101,3.565988,0.342194,35.374371,12:08
8,3.398597,3.566283,0.342647,35.384815,12:11
9,3.375563,3.568166,0.342528,35.451500,12:05


Once this is done, we save all of our model except the final layer that converts activations to probabilities of picking each token in our vocabulary. The model not including the final layer is called the *encoder*. We can save it with `save_encoder`:

In [ ]:
learn.save_encoder('finetuned')

> Jargon: Encoder: The model not including the task-specific final layer(s). It means much the same thing as *body* when applied to vision CNNs, but tends to be more used for NLP and generative models.

Before using this to fine-tune a classifier on the reviews, we can use our model to generate random reviews: since it's trained to guess what the next word of the sentence is, we can use it to write new reviews:

In [ ]:
TEXT = "I liked this movie because"
N_WORDS = 40
N_SENTENCES = 2
preds = [learn.predict(TEXT, N_WORDS, temperature=0.75) 
         for _ in range(N_SENTENCES)]

 |---------------------------------| 0.00% [0/40 00:00<?]

/Users/pengren/ws/fastai/fastai/callback/fp16.py:48: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  self.autocast,self.learn.scaler,self.scales = autocast('cuda', dtype=dtype),GradScaler('cuda', **self.kwargs),L()
/Users/pengren/ws/fastai/fastai/callback/fp16.py:48: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  self.autocast,self.learn.scaler,self.scales = autocast('cuda', dtype=dtype),GradScaler('cuda', **self.kwargs),L()


 |---------------------------------| 2.50% [1/40 00:00<00:01]

/Users/pengren/ws/fastai/fastai/callback/fp16.py:48: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  self.autocast,self.learn.scaler,self.scales = autocast('cuda', dtype=dtype),GradScaler('cuda', **self.kwargs),L()
/Users/pengren/ws/fastai/fastai/callback/fp16.py:48: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  self.autocast,self.learn.scaler,self.scales = autocast('cuda', dtype=dtype),GradScaler('cuda', **self.kwargs),L()


 |█--------------------------------| 5.00% [2/40 00:00<00:01]

...

/Users/pengren/ws/fastai/fastai/callback/fp16.py:48: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  self.autocast,self.learn.scaler,self.scales = autocast('cuda', dtype=dtype),GradScaler('cuda', **self.kwargs),L()
/Users/pengren/ws/fastai/fastai/callback/fp16.py:48: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  self.autocast,self.learn.scaler,self.scales = autocast('cuda', dtype=dtype),GradScaler('cuda', **self.kwargs),L()


 |████-----------------------------| 12.50% [5/40 00:00<00:00]

/Users/pengren/ws/fastai/fastai/callback/fp16.py:48: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  self.autocast,self.learn.scaler,self.scales = autocast('cuda', dtype=dtype),GradScaler('cuda', **self.kwargs),L()
/Users/pengren/ws/fastai/fastai/callback/fp16.py:48: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  self.autocast,self.learn.scaler,self.scales = autocast('cuda', dtype=dtype),GradScaler('cuda', **self.kwargs),L()
/Users/pengren/ws/fastai/fastai/callback/fp16.py:48: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  self.autocast,self.learn.scaler,self.scales = autocast('cuda', dtype=dtype),GradScaler('cuda', **self.kwargs),L()
/Users/pengren/ws/fastai/fastai/callback/fp16.py:48: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  self.autocast,self.learn.scaler,self.scales = autocast('cuda', dtype=dtype),GradScaler('cuda', *

 |██████████████████████████████---| 92.50% [37/40 00:00<00:00]

/Users/pengren/ws/fastai/fastai/callback/fp16.py:48: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  self.autocast,self.learn.scaler,self.scales = autocast('cuda', dtype=dtype),GradScaler('cuda', **self.kwargs),L()
/Users/pengren/ws/fastai/fastai/callback/fp16.py:48: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  self.autocast,self.learn.scaler,self.scales = autocast('cuda', dtype=dtype),GradScaler('cuda', **self.kwargs),L()
/Users/pengren/ws/fastai/fastai/callback/fp16.py:48: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  self.autocast,self.learn.scaler,self.scales = autocast('cuda', dtype=dtype),GradScaler('cuda', **self.kwargs),L()
/Users/pengren/ws/fastai/fastai/callback/fp16.py:48: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  self.autocast,self.learn.scaler,self.scales = autocast('cuda', dtype=dtype),GradScaler('cuda', *

In [ ]:
print("\n".join(preds))

i liked this movie because of the quality of We ' have been ' cool ' and funny , but i found it hard to believe there . They were too boring to be bizarre . They did n't have any practical
i liked this movie because of the humor in the first part . But i was not expecting the film . It was really hard to be working . If you were trying to get it out , i would n't have


### Training a text classifier

We can gather our data for text classification almost exactly like before:

In [ ]:
dls_clas = TextDataLoaders.from_csv(untar_data(URLs.IMDB_SAMPLE), 'texts.csv', text_col='text', label_col='label',
                                  valid_col='is_valid', text_vocab=dls_lm.vocab, bs=8)

In [ ]:
%%prompt
can you help with that?

I’ll check the CSV loader’s arguments for the filename, validation split, and shared vocabulary.The docs describe how vocabulary entries map to token IDs. I’ll check the CSV-specific parameters next.The traceback shows that `from_csv` is looking for its default filename, **`labels.csv`**, but your sample contains **`texts.csv`**. Replace the failing line with:

```python
dls_clas = TextDataLoaders.from_csv(path, 'texts.csv', text_col='text', label_col='label',
                                  valid_col='is_valid', text_vocab=dls_lm.vocab, bs=8)
```

1. **`valid_col='is_valid'`** uses the CSV’s existing validation flags. `valid='test'` belongs to the folder-based loader. ([docs.fast.ai](https://docs.fast.ai/text.data.html))
2. **`text_vocab=dls_lm.vocab`** keeps the language model’s token-to-ID mapping, so the classifier’s inputs match the encoder you trained. ([docs.fast.ai](https://docs.fast.ai/text.data.html))

```json {.tool}
{
  "id": "srv_e7993ca47274985956089d6bb5cf9e9d",
  "name": "web_search",
  "args": {
    "type": "open_page",
    "url": "https://docs.fast.ai/text.data.html"
  },
  "result": "Text data – fastai: https://docs.fast.ai/text.data.html",
  "server": true,
  "md": true
}
```

```json {.tool}
{
  "id": "srv_e9088a987d187536ef9621bd61eb35d1",
  "name": "web_search",
  "args": {
    "type": "open_page",
    "url": "https://docs.fast.ai/text.data.html"
  },
  "result": "Text data – fastai: https://docs.fast.ai/text.data.html",
  "server": true,
  "md": true
}
```

```json {.usage}
{"model": "openai/gpt-6-astra", "prompt_tokens": 80052, "completion_tokens": 925, "total_tokens": 80977, "cached_tokens": 49860, "reasoning_tokens": 679, "web_search_requests": 2, "cost": 0.6270450000000001}
```


The main difference is that we have to use the exact same vocabulary as when we were fine-tuning our language model, or the weights learned won't make any sense. We pass that vocabulary with `text_vocab`.

Then we can define our text classifier like before:

In [ ]:
learn = text_classifier_learner(dls_clas, AWD_LSTM, drop_mult=0.5, metrics=accuracy)

The difference is that before training it, we load the previous encoder:

In [ ]:
learn = learn.load_encoder('finetuned')

The last step is to train with discriminative learning rates and *gradual unfreezing*. In computer vision, we often unfreeze the model all at once, but for NLP classifiers, we find that unfreezing a few layers at a time makes a real difference.

In [ ]:
learn.fit_one_cycle(1, 2e-2)

epoch     train_loss  valid_loss  accuracy  time    
0         0.611673    0.536976    0.720000  00:13     


In just one epoch we get the same result as our training in the first section, not too bad! We can pass `-2` to `freeze_to` to freeze all except the last two parameter groups:

In [ ]:
learn.freeze_to(-2)
learn.fit_one_cycle(1, slice(1e-2/(2.6**4),1e-2))

epoch,train_loss,valid_loss,accuracy,time
0,0.247763,0.171683,0.934640,00:37


Then we can unfreeze a bit more, and continue training:

In [ ]:
learn.freeze_to(-3)
learn.fit_one_cycle(1, slice(5e-3/(2.6**4),5e-3))

epoch,train_loss,valid_loss,accuracy,time
0,0.193377,0.156696,0.941200,00:45


And finally, the whole model!

In [ ]:
learn.unfreeze()
learn.fit_one_cycle(2, slice(1e-3/(2.6**4),1e-3))

epoch,train_loss,valid_loss,accuracy,time
0,0.172888,0.153770,0.943120,01:01
1,0.161492,0.155567,0.942640,00:57
